# Experiment: SWE-bench Lite Benchmarks

This notebook shows a concise `pyflow` benchmark workflow:
- define a small SWE-bench Lite case set
- compare prompt variants with `await asyncio.gather(...)`
- launch each agent in its own case workspace
- extract token and completion stats from each `Session`
- save the full transcript and raw metrics for each run

The notebook is intentionally left unexecuted.


In [ ]:
import asyncio
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'pyflow').exists():
    REPO_ROOT = REPO_ROOT.parent.resolve()
if not (REPO_ROOT / 'pyflow').exists():
    raise RuntimeError('Start the notebook from the repo root or the demo/ directory.')

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import nest_asyncio
from IPython.display import Markdown, display

from model_setup import model
from pyflow import Agent, ParallelFailure, Request, Session, tests
from demo.benchmark_utils import (
    BenchmarkCase,
    PromptVariant,
    markdown_table,
    reset_case_workspaces,
    save_run_artifacts,
)

nest_asyncio.apply()


[03/24/26 03:08:36] INFO     Using credentials directory: /Users/goldenberg/.openhands/auth       ]8;id=646611;file:///Users/goldenberg/Developer/pyflow/.venv/lib/python3.12/site-packages/openhands/sdk/llm/auth/credentials.py\credentials.py]8;;\:]8;id=331141;file:///Users/goldenberg/Developer/pyflow/.venv/lib/python3.12/site-packages/openhands/sdk/llm/auth/credentials.py#57\57]8;;\

[03/24/26 03:08:36] INFO     Using existing OpenAI credentials                                        ]8;id=832841;file:///Users/goldenberg/Developer/pyflow/.venv/lib/python3.12/site-packages/openhands/sdk/llm/auth/openai.py\openai.py]8;;\:]8;id=402221;file:///Users/goldenberg/Developer/pyflow/.venv/lib/python3.12/site-packages/openhands/sdk/llm/auth/openai.py#659\659]8;;\

In [ ]:
WORKSPACE_ROOT = REPO_ROOT.parent / 'swebench'
ARTIFACT_ROOT = REPO_ROOT / 'demo' / 'artifacts' / 'swe-bench-lite'

base_agent = Agent(model=model)

CASES = [
    BenchmarkCase(
        instance_id='sqlfluff__sqlfluff-2419',
        repo_dir='sqlfluff__sqlfluff-2419',
        base_commit='f1dba0e1dd764ae72d67c3d5e1471cf14d3db030',
        problem=(
            'Rule L060 should report the exact offending function name instead of '
            'always mentioning both IFNULL and NVL.'
        ),
        tests=(
            'test/rules/std_L060_test.py::test__rules__std_L060_raised',
        ),
    ),
    BenchmarkCase(
        instance_id='sqlfluff__sqlfluff-1733',
        repo_dir='sqlfluff__sqlfluff-1733',
        base_commit='a1579a16b1d8913d9d7c7d12add374a290bcc78c',
        problem=(
            'Fix the extra-space regression caused by the interaction between '
            'rules L003, L036, and L039 in a WITH statement.'
        ),
        tests=(
            'test/rules/std_L003_L036_L039_combo_test.py::test__rules__std_L003_L036_L039',
            'test/rules/std_L016_L36_combo_test.py::test__rules__std_L016_L036_long_line_lint',
            'test/rules/std_L016_L36_combo_test.py::test__rules__std_L016_L036_long_line_fix',
            'test/rules/std_L016_L36_combo_test.py::test__rules__std_L016_L036_long_line_fix2',
        ),
    ),
]

VARIANTS = [
    PromptVariant(name='default'),
    PromptVariant(
        name='benchmark_prompt_v2',
        extra_instructions=(
            'Start by locating the exact failing code path, make the smallest correct '
            'patch, run the listed tests before finishing, and avoid unrelated '
            'refactors or cleanup.'
        ),
    ),
]


In [ ]:
display(
    Markdown(
        markdown_table(
            [
                {
                    'case': case.instance_id,
                    'repo_dir': case.repo_dir,
                    'tests': ', '.join(case.tests),
                }
                for case in CASES
            ]
        )
    )
)

display(
    Markdown(
        markdown_table(
            [
                {
                    'agent': variant.name,
                    'extra_instructions': variant.extra_instructions or '<default>',
                }
                for variant in VARIANTS
            ]
        )
    )
)


| case | repo_dir | tests |
| --- | --- | --- |
| sqlfluff__sqlfluff-2419 | sqlfluff__sqlfluff-2419 | test/rules/std_L060_test.py::test__rules__std_L060_raised |
| sqlfluff__sqlfluff-1733 | sqlfluff__sqlfluff-1733 | test/rules/std_L003_L036_L039_combo_test.py::test__rules__std_L003_L036_L039, test/rules/std_L016_L36_combo_test.py::test__rules__std_L016_L036_long_line_lint, test/rules/std_L016_L36_combo_test.py::test__rules__std_L016_L036_long_line_fix, test/rules/std_L016_L36_combo_test.py::test__rules__std_L016_L036_long_line_fix2 |

| agent | extra_instructions |
| --- | --- |
| default | <default> |
| benchmark_prompt_v2 | Start by locating the exact failing code path, make the smallest correct patch, run the listed tests before finishing, and avoid unrelated refactors or cleanup. |

In [4]:
def build_request(case: BenchmarkCase, variant: PromptVariant) -> Request:
    prompt = f'''
    Solve SWE-bench Lite case `{case.instance_id}`.

    Goal:
    {case.problem}
    '''.strip()

    if variant.extra_instructions:
        prompt += (
            f'\n\nAdditional benchmark guidance:\n{variant.extra_instructions}'
        )

    prompt += '\n\nVerify the fix with the attached tests before finishing.'
    return prompt >> tests(*case.tests)


async def run_case(
    case: BenchmarkCase,
    *,
    variant: PromptVariant,
    index: int,
) -> Session | ParallelFailure[BenchmarkCase]:
    agent = base_agent.replacing(workspace=WORKSPACE_ROOT / case.repo_dir)
    return await agent.run_async(build_request(case, variant))


async def run_variant(
    variant: PromptVariant,
) -> list[Session | ParallelFailure[BenchmarkCase]]:
    tasks = [
        run_case(case, variant=variant, index=index)
        for index, case in enumerate(CASES)
    ]
    results = list(await asyncio.gather(*tasks))
    save_run_artifacts(ARTIFACT_ROOT, variant.name, CASES, results)
    return results


runs: dict[str, list[Session | ParallelFailure[BenchmarkCase]]] = {}

for variant in VARIANTS:
    reset_case_workspaces(WORKSPACE_ROOT, CASES)
    runs[variant.name] = await run_variant(variant)


HEAD is now at f1dba0e1d Fix code formatting in Rule docs (#2418)
HEAD is now at a1579a16b MySQL: Add drop index support (#1738)
HEAD is now at f1dba0e1d Fix code formatting in Rule docs (#2418)
Removing test/rules/std_L060_test.py
HEAD is now at a1579a16b MySQL: Add drop index support (#1738)


[03/24/26 03:16:34] ERROR    litellm.ServiceUnavailableError: ServiceUnavailableError:           ]8;id=972459;file:///Users/goldenberg/Developer/pyflow/.venv/lib/python3.12/site-packages/openhands/sdk/llm/utils/retry_mixin.py\retry_mixin.py]8;;\:]8;id=365277;file:///Users/goldenberg/Developer/pyflow/.venv/lib/python3.12/site-packages/openhands/sdk/llm/utils/retry_mixin.py#124\124]8;;\
                             OpenAIException - upstream connect error or disconnect/reset before                   
                             headers. retried and the latest reset reason: remote connection                       
                             failure, transport failure reason: delayed connect error:                             
                             Connection refused. Attempt #1 | You can customize retry values in                    
                             the configuration.                                                                    

CancelledError: 

[03/24/26 03:23:11] ERROR    litellm.ServiceUnavailableError: ServiceUnavailableError:           ]8;id=917595;file:///Users/goldenberg/Developer/pyflow/.venv/lib/python3.12/site-packages/openhands/sdk/llm/utils/retry_mixin.py\retry_mixin.py]8;;\:]8;id=472938;file:///Users/goldenberg/Developer/pyflow/.venv/lib/python3.12/site-packages/openhands/sdk/llm/utils/retry_mixin.py#124\124]8;;\
                             OpenAIException - upstream connect error or disconnect/reset before                   
                             headers. retried and the latest reset reason: remote connection                       
                             failure, transport failure reason: delayed connect error:                             
                             Connection refused. Attempt #1 | You can customize retry values in                    
                             the configuration.                                                                    

In [ ]:
rows: list[dict[str, str]] = []

for variant in VARIANTS:
    for case, result in zip(CASES, runs[variant.name], strict=True):
        if isinstance(result, ParallelFailure):
            rows.append(
                {
                    'agent': variant.name,
                    'case': case.instance_id,
                    'status': f'failed:{result.phase}',
                    'completed': 'no',
                    'prompt_tokens': '0',
                    'completion_tokens': '0',
                    'reasoning_tokens': '0',
                    'total_tokens': '0',
                    'events': '0',
                    'notes': str(result.error),
                }
            )
            continue

        usage = result.token_usage
        rows.append(
            {
                'agent': variant.name,
                'case': case.instance_id,
                'status': result.execution_status or 'unknown',
                'completed': 'yes' if result.execution_status == 'finished' else 'no',
                'prompt_tokens': str(usage.prompt_tokens),
                'completion_tokens': str(usage.completion_tokens),
                'reasoning_tokens': str(usage.reasoning_tokens),
                'total_tokens': str(
                    usage.prompt_tokens
                    + usage.completion_tokens
                    + usage.reasoning_tokens
                ),
                'events': str(len(result.events)),
                'notes': '',
            }
        )

display(Markdown(markdown_table(rows)))

first_success = next(
    (result for result in runs[VARIANTS[0].name] if isinstance(result, Session)),
    None,
)

if first_success is None:
    {'token_usage': None, 'metrics': None}
else:
    {
        'token_usage': first_success.token_usage.model_dump(),
        'metrics': first_success.metrics.model_dump(mode='json'),
    }


In [ ]:
ARTIFACT_ROOT.resolve()
